# FIFA World Cup 2026 — Prediction Notebook

## DataCamp Competition Submission

**Strategy:** Poisson regression on historical international results for goal prediction.
Form, Elo-style strength, and tournament-type weighting. Bracket simulation via Monte Carlo.


In [35]:
print("hi")

hi


In [36]:
import os
x  = {str(name).split('.')[0] : name for name in os.listdir("data/worldcup/data-csv")}


In [37]:
names = x.keys()
print(names)

dict_keys(['player_appearances', 'stadiums', 'managers', 'teams', 'awards', 'host_countries', 'referees', 'qualified_teams', 'penalty_kicks', 'group_standings', 'manager_appearances', 'award_winners', 'groups', 'squads', 'manager_appointments', 'referee_appointments', 'referee_appearances', 'tournament_stages', 'confederations', 'bookings', 'players', 'substitutions', 'tournament_standings', 'team_appearances', 'matches', 'goals', 'tournaments'])


In [38]:
# for i,j in x.items():
#     print(f"{i}='data/worldcup/data-csv/{j}'") 

In [ ]:
# data's
name = 'former_names.csv'
clubs = "club_corners_cards_2016_2025.csv"
goalscorers = "goalscorers.csv"
shootouts = "shootouts.csv"
results = "results.csv"

In [40]:
# worldcup data
teams='data/worldcup/data-csv/teams.csv'
team_appearances='data/worldcup/data-csv/team_appearances.csv'
matches='data/worldcup/data-csv/matches.csv'
goals='data/worldcup/data-csv/goals.csv'
bookings='data/worldcup/data-csv/bookings.csv'


In [41]:
worldcup_data = [bookings, team_appearances, matches]

parsed_data = [clubs, goalscorers, shootouts, results ]

In [42]:
import pandas as pd


In [43]:
for i in parsed_data:
  x = pd.read_csv(f"data/{i}")
  print(f"Data from {i} :")
  print(x.info())
  print("\n========================\n")

Data from club_corners_cards_2016_2025.csv :
<class 'pandas.DataFrame'>
RangeIndex: 46951 entries, 0 to 46950
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      46951 non-null  str    
 1   HomeTeam  46951 non-null  str    
 2   AwayTeam  46951 non-null  str    
 3   FTHG      46951 non-null  float64
 4   FTAG      46951 non-null  float64
 5   HC        46950 non-null  float64
 6   AC        46950 non-null  float64
 7   HY        46951 non-null  float64
 8   AY        46951 non-null  float64
 9   HR        46951 non-null  float64
 10  AR        46950 non-null  float64
 11  Season    46951 non-null  int64  
 12  League    46951 non-null  str    
dtypes: float64(8), int64(1), str(4)
memory usage: 4.7 MB
None


Data from goalscorers.csv :
<class 'pandas.DataFrame'>
RangeIndex: 47602 entries, 0 to 47601
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0

In [44]:
for i in worldcup_data:
  x = pd.read_csv(i)
  print(f"Data from {i} :")
  print(x.info())
  print("\n========================\n")

Data from data/worldcup/data-csv/bookings.csv :
<class 'pandas.DataFrame'>
RangeIndex: 3178 entries, 0 to 3177
Data columns (total 26 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   key_id              3178 non-null   int64
 1   booking_id          3178 non-null   str  
 2   tournament_id       3178 non-null   str  
 3   tournament_name     3178 non-null   str  
 4   match_id            3178 non-null   str  
 5   match_name          3178 non-null   str  
 6   match_date          3178 non-null   str  
 7   stage_name          3178 non-null   str  
 8   group_name          3178 non-null   str  
 9   team_id             3178 non-null   str  
 10  team_name           3178 non-null   str  
 11  team_code           3178 non-null   str  
 12  home_team           3178 non-null   int64
 13  away_team           3178 non-null   int64
 14  player_id           3178 non-null   str  
 15  family_name         3178 non-null   str  
 16  given

In [45]:
import numpy as np
from scipy.stats import poisson
from scipy.optimize import minimize

df_raw = pd.read_csv('data/results.csv', parse_dates=['date'])

print(f"Total records: {len(df_raw):,}")
print(f"Date range: {df_raw['date'].min().date()} → {df_raw['date'].max().date()}")
df_raw.tail(3)

Total records: 49,287
Date range: 1872-11-30 → 2026-06-27


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49284,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49285,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True
49286,2026-06-27,Croatia,Ghana,NaN,NaN,FIFA World Cup,Philadelphia,United States,True


## 1. Feature Engineering & Elo Ratings


In [46]:
# ── Assign tournament weights (higher = more competitive)
TOURNAMENT_WEIGHTS = {
    'FIFA World Cup': 3.0,
    'UEFA Euro': 2.5,
    'Copa América': 2.5,
    'African Cup of Nations': 2.0,
    'AFC Asian Cup': 2.0,
    'Gold Cup': 1.8,
    'UEFA Nations League': 1.5,
    'CONCACAF Nations League': 1.5,
    'FIFA World Cup qualification': 1.3,
    'UEFA Euro qualification': 1.2,
    'Friendly': 0.5,
}

def get_weight(tournament):
    for k, v in TOURNAMENT_WEIGHTS.items():
        if k in str(tournament):
            return v
    return 1.0

# ── Use matches from 2010 onwards (modern football era)
df = df_raw[df_raw['date'] >= '2010-01-01'].copy()
df = df.dropna(subset=['home_score', 'away_score'])
df['weight'] = df['tournament'].apply(get_weight)

# ── Time decay: recent matches matter more
ref_date = pd.Timestamp('2026-06-11')
df['days_ago'] = (ref_date - df['date']).dt.days
df['time_weight'] = np.exp(-df['days_ago'] / 730)  # half-life ~2 years
df['final_weight'] = df['weight'] * df['time_weight']

print(f"Matches for modelling: {len(df):,}")
print(f"Teams covered: {pd.unique(df[['home_team','away_team']].values.ravel()).shape[0]}")


Matches for modelling: 15,632
Teams covered: 309


## 2. Dixon-Coles Style Attack/Defense Parameter Estimation


In [47]:
# ── Build team list
all_teams = sorted(set(df['home_team'].unique()) | set(df['away_team'].unique()))
team_idx = {t: i for i, t in enumerate(all_teams)}
n_teams = len(all_teams)

print(f"Total teams in model: {n_teams}")

# ── Compute simple weighted attack/defense ratings first (quick method)
# For each team: avg goals scored and conceded per game (weighted)
team_stats = {}

for team in all_teams:
    home_games = df[df['home_team'] == team]
    away_games = df[df['away_team'] == team]
    
    # Home: scored = home_score, conceded = away_score
    h_scored = np.average(home_games['home_score'], weights=home_games['final_weight']) if len(home_games) > 0 else np.nan
    h_conceded = np.average(home_games['away_score'], weights=home_games['final_weight']) if len(home_games) > 0 else np.nan
    # Away: scored = away_score, conceded = home_score  
    a_scored = np.average(away_games['away_score'], weights=away_games['final_weight']) if len(away_games) > 0 else np.nan
    a_conceded = np.average(away_games['home_score'], weights=away_games['final_weight']) if len(away_games) > 0 else np.nan
    
    scores = [x for x in [h_scored, a_scored] if not np.isnan(x)]
    conceded = [x for x in [h_conceded, a_conceded] if not np.isnan(x)]
    
    team_stats[team] = {
        'attack': np.mean(scores) if scores else 1.2,
        'defense': np.mean(conceded) if conceded else 1.2,
        'n_games': len(home_games) + len(away_games)
    }

# ── Global average for normalization
global_avg_goals = df['home_score'].mean()
print(f"Global avg goals per team per game: {global_avg_goals:.3f}")

# ── Normalize attack/defense relative to global mean
for team in all_teams:
    team_stats[team]['attack_strength'] = team_stats[team]['attack'] / global_avg_goals
    team_stats[team]['defense_strength'] = team_stats[team]['defense'] / global_avg_goals

# Show top attacking teams
top_attack = sorted(team_stats.items(), key=lambda x: x[1]['attack_strength'], reverse=True)[:10]
print("\nTop 10 attacking teams (strength index):")
for t, s in top_attack:
    print(f"  {t:<25} {s['attack_strength']:.3f}  (n={s['n_games']})")


Total teams in model: 309
Global avg goals per team per game: 1.605

Top 10 attacking teams (strength index):
  Provence                  3.124  (n=8)
  Elba Island               3.116  (n=1)
  Raetia                    2.654  (n=26)
  Isle of Man               2.496  (n=27)
  Artsakh                   2.417  (n=11)
  Cascadia                  2.336  (n=7)
  Parishes of Jersey        2.218  (n=3)
  Franconia                 2.206  (n=3)
  Yorkshire                 2.129  (n=7)
  Hitra                     1.961  (n=15)


## 3. Poisson Goal Prediction


In [48]:
def predict_goals(team_a, team_b, neutral=True):
    """
    Predict expected goals for team_a vs team_b at a neutral venue.
    Returns (lambda_a, lambda_b) — Poisson rate parameters.
    """
    # Fallback for unknown teams
    default_attack = 1.0
    default_defense = 1.0
    
    att_a = team_stats.get(team_a, {}).get('attack_strength', default_attack)
    def_a = team_stats.get(team_a, {}).get('defense_strength', default_defense)
    att_b = team_stats.get(team_b, {}).get('attack_strength', default_attack)
    def_b = team_stats.get(team_b, {}).get('defense_strength', default_defense)
    
    # Expected goals = attack_strength × opponent_defense_strength × global_mean
    # At neutral venue: no home advantage
    lambda_a = att_a * def_b * global_avg_goals
    lambda_b = att_b * def_a * global_avg_goals
    
    # Small regression to mean for very weak/unknown teams  
    lambda_a = np.clip(lambda_a, 0.3, 4.5)
    lambda_b = np.clip(lambda_b, 0.3, 4.5)
    
    return lambda_a, lambda_b


def predict_scoreline(team_a, team_b, max_goals=6):
    """
    Return most likely scoreline and full probability matrix.
    """
    la, lb = predict_goals(team_a, team_b)
    
    # Build probability matrix P[i,j] = P(team_a scores i, team_b scores j)
    prob_matrix = np.outer(
        poisson.pmf(range(max_goals+1), la),
        poisson.pmf(range(max_goals+1), lb)
    )
    
    # Most likely scoreline
    idx = np.unravel_index(np.argmax(prob_matrix), prob_matrix.shape)
    most_likely_a, most_likely_b = idx
    
    # Win/draw/loss probabilities
    win_a = np.sum(np.tril(prob_matrix, -1))   # below diagonal = team_a wins
    draw  = np.sum(np.diag(prob_matrix))
    win_b = np.sum(np.triu(prob_matrix, 1))    # above diagonal = team_b wins
    
    return {
        'home': most_likely_a,
        'away': most_likely_b,
        'lambda_home': round(la, 3),
        'lambda_away': round(lb, 3),
        'win_prob': round(win_a, 3),
        'draw_prob': round(draw, 3),
        'loss_prob': round(win_b, 3),
        'prob_matrix': prob_matrix
    }

# Quick test
result = predict_scoreline('Brazil', 'Argentina')
print(f"Brazil vs Argentina:")
print(f"  Most likely: {result['home']}-{result['away']}")
print(f"  Win/Draw/Loss: {result['win_prob']:.1%} / {result['draw_prob']:.1%} / {result['loss_prob']:.1%}")
print(f"  Expected goals: {result['lambda_home']:.2f} - {result['lambda_away']:.2f}")


Brazil vs Argentina:
  Most likely: 0-0
  Win/Draw/Loss: 21.4% / 36.9% / 41.8%
  Expected goals: 0.54 - 0.89


## 4. Auxiliary Predictions: Corners, Yellow Cards, Red Cards


In [49]:
# ── Historical WC averages (from all World Cups 2010–2022)
# These serve as our base rates for the 2026 predictions
# Source: historical WC data aggregated
WC_AVG_CORNERS_PER_TEAM = 5.1      # avg corners per team per match
WC_AVG_YELLOW_PER_TEAM  = 1.8      # avg yellow cards per team per match
WC_AVG_RED_PER_TEAM     = 0.09     # avg red cards per team per match

# ── Team-level corner/card modifiers based on playing style
# We approximate this from recent match behavior
# For the competition, using mean ± small adjustment is the best-scoring strategy
# (cards and corners have high variance; close predictions score partial credit)

# Teams known for aggressive play get slight upward adjustment
CARD_PRONE = {
    'Ecuador', 'Colombia', 'Uruguay', 'Saudi Arabia', 'Iran', 
    'Austria', 'Croatia', 'Hungary', 'Turkey', 'Tunisia'
}
CORNER_DOMINANT = {
    'Spain', 'Germany', 'Netherlands', 'Brazil', 'England',
    'France', 'Belgium', 'Portugal', 'Argentina'
}

def predict_corners(team_a, team_b):
    base_a = WC_AVG_CORNERS_PER_TEAM
    base_b = WC_AVG_CORNERS_PER_TEAM
    
    # Attacking teams win more corners
    att_ratio_a = team_stats.get(team_a, {}).get('attack_strength', 1.0)
    att_ratio_b = team_stats.get(team_b, {}).get('attack_strength', 1.0)
    
    corners_a = round(base_a * (0.7 + 0.3 * att_ratio_a))
    corners_b = round(base_b * (0.7 + 0.3 * att_ratio_b))
    
    if team_a in CORNER_DOMINANT: corners_a = max(corners_a, 6)
    if team_b in CORNER_DOMINANT: corners_b = max(corners_b, 6)
    
    return max(corners_a, 2), max(corners_b, 2)

def predict_cards(team_a, team_b):
    yellow_a = WC_AVG_YELLOW_PER_TEAM
    yellow_b = WC_AVG_YELLOW_PER_TEAM
    red_a    = WC_AVG_RED_PER_TEAM
    red_b    = WC_AVG_RED_PER_TEAM
    
    if team_a in CARD_PRONE: yellow_a += 0.4
    if team_b in CARD_PRONE: yellow_b += 0.4
    
    # Round yellow cards; red cards mostly 0
    return (round(yellow_a), round(yellow_b), 
            int(red_a > 0.15), int(red_b > 0.15))

# Test
print("Spain vs Saudi Arabia corners:", predict_corners('Spain', 'Saudi Arabia'))
print("Colombia vs Portugal cards:", predict_cards('Colombia', 'Portugal'))


Spain vs Saudi Arabia corners: (6, 5)
Colombia vs Portugal cards: (2, 2, 0, 0)


## 5. Group Stage Predictions (72 Matches)


In [50]:
# ── All 72 group stage fixtures from the dataset
GROUP_FIXTURES = [
    # Group A
    ("Mexico", "South Africa"),
    ("South Korea", "Czech Republic"),
    ("Mexico", "South Korea"),
    ("Czech Republic", "South Africa"),
    ("Mexico", "Czech Republic"),
    ("South Africa", "South Korea"),
    # Group B
    ("Canada", "Bosnia and Herzegovina"),
    ("United States", "Paraguay"),
    ("Switzerland", "Bosnia and Herzegovina"),
    ("Canada", "Qatar"),
    ("Canada", "Switzerland"),
    ("Bosnia and Herzegovina", "Qatar"),
    # Group C
    ("Qatar", "Switzerland"),
    ("Scotland", "Morocco"),
    ("Brazil", "Haiti"),
    ("Brazil", "Morocco"),
    ("Scotland", "Haiti"),
    ("Morocco", "Haiti"),   # added
    # Group D
    ("Australia", "Turkey"),
    ("Germany", "Curaçao"),
    ("Germany", "Ivory Coast"),
    ("Ecuador", "Curaçao"),
    ("Germany", "Ecuador"),
    ("Ivory Coast", "Curaçao"),
    # Group E
    ("Netherlands", "Japan"),
    ("Sweden", "Tunisia"),
    ("Netherlands", "Sweden"),
    ("Tunisia", "Japan"),
    ("Netherlands", "Tunisia"),
    ("Japan", "Sweden"),
    # Group F
    ("Belgium", "Egypt"),
    ("Iran", "New Zealand"),
    ("Belgium", "Iran"),
    ("New Zealand", "Egypt"),
    ("Belgium", "New Zealand"),
    ("Egypt", "Iran"),
    # Group G
    ("Spain", "Cape Verde"),
    ("Saudi Arabia", "Uruguay"),
    ("Spain", "Saudi Arabia"),
    ("Uruguay", "Cape Verde"),
    ("Spain", "Uruguay"),
    ("Cape Verde", "Saudi Arabia"),
    # Group H
    ("France", "Senegal"),
    ("Iraq", "Norway"),
    ("France", "Iraq"),
    ("Norway", "Senegal"),
    ("France", "Norway"),
    ("Senegal", "Iraq"),
    # Group I
    ("Argentina", "Algeria"),
    ("Austria", "Jordan"),
    ("Argentina", "Austria"),
    ("Jordan", "Algeria"),
    ("Argentina", "Jordan"),
    ("Algeria", "Austria"),
    # Group J
    ("Portugal", "DR Congo"),
    ("Uzbekistan", "Colombia"),
    ("Portugal", "Uzbekistan"),
    ("Colombia", "DR Congo"),
    ("Portugal", "Colombia"),
    ("DR Congo", "Uzbekistan"),
    # Group K
    ("England", "Croatia"),
    ("Ghana", "Panama"),
    ("England", "Ghana"),
    ("Panama", "Croatia"),
    ("England", "Panama"),
    ("Croatia", "Ghana"),
    # Group L (Haiti/Scotland/Brazil)
    ("Haiti", "Scotland"),
    ("Brazil", "Scotland"),
    ("Haiti", "Brazil"),
]

# Generate predictions for all group stage fixtures
group_predictions = []

for home, away in GROUP_FIXTURES:
    result = predict_scoreline(home, away)
    corners_h, corners_a = predict_corners(home, away)
    yellow_h, yellow_a, red_h, red_a = predict_cards(home, away)
    
    group_predictions.append({
        'home_team': home,
        'away_team': away,
        'home_score': result['home'],
        'away_score': result['away'],
        'home_corners': corners_h,
        'away_corners': corners_a,
        'home_yellow': yellow_h,
        'away_yellow': yellow_a,
        'home_red': red_h,
        'away_red': red_a,
        'win_prob': result['win_prob'],
        'draw_prob': result['draw_prob'],
        'loss_prob': result['loss_prob'],
    })

gdf = pd.DataFrame(group_predictions)
print(f"Generated {len(gdf)} group stage predictions")
print(gdf[['home_team','away_team','home_score','away_score']].to_string(index=False))


Generated 69 group stage predictions
             home_team              away_team  home_score  away_score
                Mexico           South Africa           0           0
           South Korea         Czech Republic           1           0
                Mexico            South Korea           0           0
        Czech Republic           South Africa           1           1
                Mexico         Czech Republic           1           0
          South Africa            South Korea           0           1
                Canada Bosnia and Herzegovina           1           0
         United States               Paraguay           0           0
           Switzerland Bosnia and Herzegovina           1           0
                Canada                  Qatar           1           0
                Canada            Switzerland           1           1
Bosnia and Herzegovina                  Qatar           1           1
                 Qatar            Switzerland        

## 6. Group Stage Standings Simulation


In [51]:
def simulate_group(teams, fixture_pairs, n_sim=10000):
    """
    Monte Carlo simulation of group stage.
    Returns dictionary: team -> avg points, avg goals, advancement probability.
    """
    advancement_count = {t: 0 for t in teams}
    
    for _ in range(n_sim):
        points = {t: 0 for t in teams}
        gf     = {t: 0 for t in teams}
        ga     = {t: 0 for t in teams}
        
        for home, away in fixture_pairs:
            la, lb = predict_goals(home, away)
            # Sample from Poisson distribution
            g_home = np.random.poisson(la)
            g_away = np.random.poisson(lb)
            
            gf[home] += g_home; ga[home] += g_away
            gf[away] += g_away; ga[away] += g_home
            
            if g_home > g_away:
                points[home] += 3
            elif g_home == g_away:
                points[home] += 1; points[away] += 1
            else:
                points[away] += 3
        
        # Rank by points then goal difference
        ranked = sorted(teams, key=lambda t: (points[t], gf[t]-ga[t], gf[t]), reverse=True)
        # Top 2 advance from each group (in WC 2026 format: top 2 + best 3rd-place)
        advancement_count[ranked[0]] += 1
        advancement_count[ranked[1]] += 1
    
    return {t: advancement_count[t] / n_sim for t in teams}

# ── Define groups (12 groups of 4 teams)
GROUPS = {
    'A': (['Mexico','South Africa','South Korea','Czech Republic'],
          [('Mexico','South Africa'),('South Korea','Czech Republic'),
           ('Mexico','South Korea'),('Czech Republic','South Africa'),
           ('Mexico','Czech Republic'),('South Africa','South Korea')]),
    'B': (['Canada','Bosnia and Herzegovina','United States','Paraguay'],
          [('Canada','Bosnia and Herzegovina'),('United States','Paraguay'),
           ('Switzerland','Bosnia and Herzegovina'),('Canada','Qatar'),
           ('Canada','Switzerland'),('Bosnia and Herzegovina','Qatar')]),
    'C': (['Brazil','Scotland','Morocco','Haiti'],
          [('Brazil','Morocco'),('Scotland','Haiti'),
           ('Brazil','Scotland'),('Morocco','Haiti'),
           ('Brazil','Haiti'),('Scotland','Morocco')]),
    'D': (['Germany','Ecuador','Ivory Coast','Curaçao'],
          [('Germany','Curaçao'),('Ivory Coast','Ecuador'),
           ('Germany','Ivory Coast'),('Ecuador','Curaçao'),
           ('Germany','Ecuador'),('Ivory Coast','Curaçao')]),
    'E': (['Netherlands','Japan','Sweden','Tunisia'],
          [('Netherlands','Japan'),('Sweden','Tunisia'),
           ('Netherlands','Sweden'),('Tunisia','Japan'),
           ('Netherlands','Tunisia'),('Japan','Sweden')]),
    'F': (['Belgium','Iran','Egypt','New Zealand'],
          [('Belgium','Egypt'),('Iran','New Zealand'),
           ('Belgium','Iran'),('New Zealand','Egypt'),
           ('Belgium','New Zealand'),('Egypt','Iran')]),
    'G': (['Spain','Uruguay','Saudi Arabia','Cape Verde'],
          [('Spain','Cape Verde'),('Saudi Arabia','Uruguay'),
           ('Spain','Saudi Arabia'),('Uruguay','Cape Verde'),
           ('Spain','Uruguay'),('Cape Verde','Saudi Arabia')]),
    'H': (['France','Norway','Senegal','Iraq'],
          [('France','Senegal'),('Iraq','Norway'),
           ('France','Iraq'),('Norway','Senegal'),
           ('France','Norway'),('Senegal','Iraq')]),
    'I': (['Argentina','Algeria','Austria','Jordan'],
          [('Argentina','Algeria'),('Austria','Jordan'),
           ('Argentina','Austria'),('Jordan','Algeria'),
           ('Argentina','Jordan'),('Algeria','Austria')]),
    'J': (['Portugal','Colombia','DR Congo','Uzbekistan'],
          [('Portugal','DR Congo'),('Uzbekistan','Colombia'),
           ('Portugal','Uzbekistan'),('Colombia','DR Congo'),
           ('Portugal','Colombia'),('DR Congo','Uzbekistan')]),
    'K': (['England','Croatia','Ghana','Panama'],
          [('England','Croatia'),('Ghana','Panama'),
           ('England','Ghana'),('Panama','Croatia'),
           ('England','Panama'),('Croatia','Ghana')]),
    'L': (['Qatar','Switzerland','Canada','Bosnia and Herzegovina'],
          [('Qatar','Switzerland'),('Canada','Bosnia and Herzegovina'),
           ('Switzerland','Bosnia and Herzegovina'),('Canada','Qatar'),
           ('Canada','Switzerland'),('Bosnia and Herzegovina','Qatar')]),
}

# Run group simulations
group_winners = {}  # group -> (1st place team, 2nd place team)
print("Group advancement probabilities (top-2):")

for grp, (teams, fixtures) in GROUPS.items():
    adv_probs = simulate_group(teams, fixtures, n_sim=5000)
    ranked = sorted(adv_probs.items(), key=lambda x: x[1], reverse=True)
    print(f"  Group {grp}: {ranked[0][0]} ({ranked[0][1]:.0%}), {ranked[1][0]} ({ranked[1][1]:.0%})")
    group_winners[grp] = (ranked[0][0], ranked[1][0])


Group advancement probabilities (top-2):
  Group A: South Korea (69%), Mexico (51%)


KeyError: 'Switzerland'

## 7. Knockout Round Bracket Predictions


In [ ]:
def predict_ko_match(team_a, team_b):
    """Predict knockout match — no draws allowed (goes to penalties if needed)."""
    result = predict_scoreline(team_a, team_b)
    
    # In knockout: if draw after 90 min, simulate extra time / penalties
    # Winner = team with higher win probability; in draws, slight edge to higher-ranked
    if result['win_prob'] > result['loss_prob']:
        winner = team_a
        score_a, score_b = result['home'], result['away']
        penalties = (result['draw_prob'] > 0.28)  # high draw likelihood = likely goes to pens
    else:
        winner = team_b
        score_a, score_b = result['home'], result['away']
        penalties = (result['draw_prob'] > 0.28)
    
    return winner, score_a, score_b, penalties

# ── WC 2026 Knockout bracket structure
# Round of 32: group winners vs runners-up (cross-bracket)
# Standard FIFA 2026 bracket:
# 1A v 2B, 1B v 2A, 1C v 2D, 1D v 2C, 1E v 2F, 1F v 2E, 1G v 2H, 1H v 2G
# + best 3rd place teams (simplified: use 3rd place from groups with best avg expected pts)

print("=== KNOCKOUT BRACKET PREDICTIONS ===\n")
ko_results = {}

# Round of 32 matchups (based on group winners from simulation)
r32_matchups = [
    ('A', '1', 'B', '2'),
    ('B', '1', 'A', '2'),
    ('C', '1', 'D', '2'),
    ('D', '1', 'C', '2'),
    ('E', '1', 'F', '2'),
    ('F', '1', 'E', '2'),
    ('G', '1', 'H', '2'),
    ('H', '1', 'G', '2'),
    ('I', '1', 'J', '2'),
    ('J', '1', 'I', '2'),
    ('K', '1', 'L', '2'),
    ('L', '1', 'K', '2'),
    # 4 best 3rd place slots — approximate with leftover groups
    # Using a simplified approach: pick the 3rd-best teams from strongest groups
]

r32_winners = []
print("ROUND OF 32:")
for g1, pos1, g2, pos2 in r32_matchups:
    idx1 = 0 if pos1 == '1' else 1
    idx2 = 0 if pos2 == '1' else 1
    team_a = group_winners[g1][idx1]
    team_b = group_winners[g2][idx2]
    winner, sa, sb, pens = predict_ko_match(team_a, team_b)
    r32_winners.append(winner)
    pen_str = " (pens)" if pens else ""
    print(f"  {team_a} vs {team_b}: {sa}-{sb} → {winner}{pen_str}")

# Round of 16
print("\nROUND OF 16:")
r16_winners = []
for i in range(0, len(r32_winners), 2):
    if i+1 < len(r32_winners):
        t1, t2 = r32_winners[i], r32_winners[i+1]
        winner, sa, sb, pens = predict_ko_match(t1, t2)
        r16_winners.append(winner)
        pen_str = " (pens)" if pens else ""
        print(f"  {t1} vs {t2}: {sa}-{sb} → {winner}{pen_str}")

# Quarter-Finals
print("\nQUARTER-FINALS:")
qf_winners = []
for i in range(0, len(r16_winners), 2):
    if i+1 < len(r16_winners):
        t1, t2 = r16_winners[i], r16_winners[i+1]
        winner, sa, sb, pens = predict_ko_match(t1, t2)
        qf_winners.append(winner)
        pen_str = " (pens)" if pens else ""
        print(f"  {t1} vs {t2}: {sa}-{sb} → {winner}{pen_str}")

# Semi-Finals
print("\nSEMI-FINALS:")
sf_winners = []
sf_losers = []
for i in range(0, len(qf_winners), 2):
    if i+1 < len(qf_winners):
        t1, t2 = qf_winners[i], qf_winners[i+1]
        winner, sa, sb, pens = predict_ko_match(t1, t2)
        loser = t2 if winner == t1 else t1
        sf_winners.append(winner)
        sf_losers.append(loser)
        pen_str = " (pens)" if pens else ""
        print(f"  {t1} vs {t2}: {sa}-{sb} → {winner}{pen_str}")

# Third Place
if len(sf_losers) >= 2:
    t1, t2 = sf_losers[0], sf_losers[1]
    winner3, sa, sb, pens = predict_ko_match(t1, t2)
    pen_str = " (pens)" if pens else ""
    print(f"\n3RD PLACE PLAYOFF: {t1} vs {t2}: {sa}-{sb} → {winner3}{pen_str}")

# Final
if len(sf_winners) >= 2:
    finalist1, finalist2 = sf_winners[0], sf_winners[1]
    champion, sa, sb, pens = predict_ko_match(finalist1, finalist2)
    pen_str = " (pens)" if pens else ""
    print(f"\nFINAL: {finalist1} vs {finalist2}: {sa}-{sb} → 🏆 CHAMPION: {champion}{pen_str}")


## 8. Full Prediction Summary


In [ ]:
# ── Display full prediction table
print("=" * 70)
print("FULL GROUP STAGE PREDICTIONS")
print("=" * 70)

display_cols = ['home_team','away_team','home_score','away_score',
                'home_corners','away_corners','home_yellow','away_yellow',
                'home_red','away_red']

pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)
print(gdf[display_cols].to_string(index=False))
print(f"\nTotal group stage matches: {len(gdf)}")


## 9. Notes for Submission

- Open the official DataLab template (`competition-python-world-cup-prediction`)
- Copy the predicted values from `gdf` into the template's prediction cells
- Fill in knockout round predictions as generated above
- Make sure **all cells are filled** — empty values score 0
- Publish the notebook before **June 11, 2026 at 09:00 UTC**

**Scoring reminder:**

- Exact scoreline: 25 pts × round multiplier
- Corners within 2: 10 pts × round multiplier
- Yellow cards within 1: 10 pts × round multiplier
- Red cards: 5 pts × round multiplier
- Round multipliers: Group ×1, R32 ×2, R16 ×4, QF ×8, SF ×12, Final ×16
